In [1]:
from tdc.multi_pred import DDI

data = DDI(name="DrugBank", path="/media/onlyzabao/Data/TDC")

Downloading...
100%|██████████| 44.4M/44.4M [00:21<00:00, 2.05MiB/s]
Loading...
Done!


In [4]:
import pandas as pd

df = data.get_data()

label_counts = df["Y"].value_counts()

print(label_counts)

Y
49    60751
47    34360
73    23779
75     9470
60     8397
      ...  
28       11
1        11
52       10
26        7
42        6
Name: count, Length: 86, dtype: int64


In [7]:
from tdc.utils import get_label_map

label = get_label_map(name="DrugBank", task="DDI", path="/media/onlyzabao/Data/TDC")

print(label[49])

The risk or severity of adverse effects can be increased when #Drug1 is combined with #Drug2.


In [31]:
import json
from sklearn.model_selection import train_test_split


reduced_df = df[df["Y"] == 49]

with open("datasets/Hetionet/vocab/entity_vocab.json", "r") as file:
    entity = json.load(file)

mask = reduced_df.apply(
    lambda row: ("Compound::" + row["Drug1_ID"]) in entity
    and ("Compound::" + row["Drug2_ID"]) in entity,
    axis=1,
)
hetion_df = reduced_df[mask]

train_hetion_df, test_hetion_df = train_test_split(hetion_df, test_size=0.3)
test_hetion_df, val_hetion_df = train_test_split(test_hetion_df, test_size=0.3)

print("Train: ", train_hetion_df.shape[0])
print("Test: ", test_hetion_df.shape[0])
print("Validate: ", val_hetion_df.shape[0])

Train:  32594
Test:  9779
Validate:  4191


In [37]:
with open('datasets/Hetionet/test.txt', 'w') as file:
    for index, row in test_hetion_df.iterrows():
        compound_1 = 'Compound::' + row['Drug1_ID']
        compound_2 = 'Compound::' + row['Drug2_ID']
        file.write(compound_1 + '\t' + 'CiC' + '\t' + compound_2 + '\n')
